# ICCIT 2026 — quantization x language: P0 grid

Run order in this notebook is deliberate. Do not skip ahead.

1. **Settings check** — Accelerator must be **GPU T4 x2** and **Internet ON**.
   A P100 session cannot run INT8 or NF4 at all; cell 2 will stop you.
2. **Probe** — records the exact GPU and library versions for the paper.
3. **Smoke** — 20 items x 5 languages x 3 precisions, ~10 min. Proves the whole
   path works before a long session is spent on it.
4. **P0** — the full 900-item grid.
5. **Save** — Save Version (Save & Run All) so the outputs persist, then pull
   them down locally with `kaggle kernels output`.

Everything that defines the experiment lives in `configs/experiment.yaml` and
`configs/item_id_manifest.json` in the repo. Nothing is configured here.

In [ ]:
# 1. Get the code. Push D:\quantlang to this repo first.
REPO_URL = "https://github.com/fairuz-anadi/quantization.git"
REF      = "main"          # pin to a commit SHA once the run is the real one

import os, subprocess, sys
WORK = "/kaggle/working"
SRC  = f"{WORK}/quantlang"

if not os.path.exists(SRC):
    subprocess.run(["git", "clone", "--depth", "1", "-b", REF, REPO_URL, SRC], check=True)
print(subprocess.run(["git", "-C", SRC, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip())

os.chdir(SRC)
sys.path.insert(0, SRC)

In [ ]:
# 2. Dependencies. Kaggle's torch is CUDA-matched -- never reinstall it.
!pip install -q -U "transformers>=4.45" "bitsandbytes>=0.43" accelerate datasets pyyaml

In [ ]:
# 3. Environment probe. STOP HERE if this exits non-zero.
!python scripts/probe_env.py --outdir /kaggle/working

In [ ]:
# 4. Smoke test: 20 items, every language, every precision.
# Output goes to results/smoke/, which is gitignored and can never reach a table.
!python scripts/run_eval.py --all-precisions --limit 20 \
    --outdir results/smoke/kaggle --tag smoke --store-prompts

### Check the smoke output before continuing

- `acc` should be plainly above 0.25 for English. If every language sits at
  chance, something is wrong with the prompt or the option token ids -- stop.
- `truncated=0` is expected at 4096 tokens. A non-zero count for Sinhala or
  Assamese is worth knowing about before the full run.
- The run **crashes** rather than continues if a quantizer silently failed to
  apply, so reaching this point means INT8 and NF4 really are quantized.

Only when all three look right, run the full grid.

In [ ]:
# 5. P0 -- the full 900-item grid, all five languages, all three precisions.
# Precision is the outer loop so every language in a precision shares one model
# load in one session, which is what makes the latency column comparable.
# Estimated ~2h 15m on T4 for 5 languages.
!python scripts/run_eval.py --all-precisions \
    --outdir /kaggle/working/raw --tag p0

In [ ]:
# 6. Inventory: every cell must read 900/900 before these results are used.
!python scripts/build_tidy.py --indir /kaggle/working/raw --inventory

In [ ]:
# 7. Package the raw output for download.
import shutil
shutil.make_archive("/kaggle/working/p0_raw", "zip", "/kaggle/working/raw")
print(sorted(os.listdir("/kaggle/working")))

### Then, locally

```bash
kaggle kernels output <user>/<kernel-slug> -p results/raw/
python scripts/build_tidy.py
python scripts/analyze.py
```

`results/raw/` is append-only and is the provenance for every number in the
paper. Nothing in the repo writes to it except that `kaggle kernels output`
command.